In [1]:
# Install dependencies jika belum
!pip install gdown pandas psycopg2-binary

In [2]:
# Import modul buatan kita sendiri
from pyspark.sql import functions as F
from ingestion.downloader import download_drive_dataset
from ingestion.spark_client import get_spark_session
from ingestion.etl_bronze import seed_postgresql, load_to_minio_bronze

# 1. Download Data
download_drive_dataset()

# 2. Nyalakan Spark
spark = get_spark_session("Bronze-Ingestion-Pipeline")
print("\nSpark Session Berhasil Dibuat!")

# 3. Pastikan Data Customers sudah tersedia di PostgreSQL
seed_postgresql(spark)

# 4. Ingest semua data (dari Postgres & CSV) ke MinIO (Bronze Layer)
load_to_minio_bronze(spark)

# 5. Validasi jumlah final sales di bronze dan distribusi year-month
sales_bronze_df = spark.read.parquet("s3a://bronze/sales")
print(f"\nTotal baris sales di bronze: {sales_bronze_df.count():,}")
display(
    sales_bronze_df
    .groupBy(F.date_format("invoice_date", "yyyy-MM").alias("year_month"))
    .count()
    .orderBy("year_month")
)

# 6. Tutup koneksi dengan baik
spark.stop()

Data belum lengkap. Memulai proses download dari folder Google Drive...
File yang belum ada: ['customers.csv', 'items.csv', 'sales_1m.csv']
Data berhasil didownload dan disimpan di: /home/jovyan/work/data/dataset
File yang tersedia: ['customers.csv', 'items.csv', 'sales_1m.csv']

Spark Session Berhasil Dibuat!

[Step 1] Mengecek tabel 'customers' di PostgreSQL...
-> Tabel 'customers' siap dipakai di PostgreSQL dengan 500000 baris.

[Step 2] Memulai proses Data Ingestion ke MinIO (Layer Bronze)...
  -> Menarik tabel 'customers' dari PostgreSQL...
  -> Membaca 'sales_1m.csv' dengan Explicit Schema...
  -> Sukses disimpan di MinIO: s3a://bronze/sales
  -> Membaca 'items.csv'...

🎉 SELAMAT! Pipeline Ingestion Tahap 1 (Bronze Layer) Selesai!

Total baris sales di bronze: 998,224


DataFrame[year_month: string, count: bigint]